# Model training

## Imports

In [15]:
import os
import numpy as np
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Hide Warnings

## Directories

In [16]:
base_dir = os.getcwd()
processed_data_dir = os.path.join(base_dir, 'processed_data')

model_dir = os.path.join(base_dir, 'models')
if os.path.exists: os.makedirs(model_dir, exist_ok=True)

## Load data

In [17]:
dataset_path = os.path.join(processed_data_dir, "ampds_dataset.npz")

with np.load(dataset_path) as saved:
    data = {key: saved[key] for key in saved.files}

scaling_factors = {k: data[k] for k in ("x_min", "x_max", "y_min", "y_max")}
idx_dict = {
    split: (data[f"{split}_inp"], data[f"{split}_out"])
    for split in ("train", "val", "test")}

## Make PQ image

In [18]:
def make_image(x_win):
    
    x_win = tf.convert_to_tensor(x_win, dtype=tf.float32)
    p, q = x_win[:,0], x_win[:,1]
    
    # Build PQ Signature
    p_col, q_row = tf.reshape(p, (-1,1)), tf.reshape(q, (1, -1))
    S_xy = tf.sqrt(p_col**2 + q_row**2) # Compute pairwise S_xy, producing (W,W) matrix.
    top = tf.concat([tf.zeros((1,1)), q_row], axis=1) # Build first row (q only)
    left = tf.concat([p_col, S_xy], axis=1) # Build remaining rows. First column is p only.
    S = tf.concat([top, left], axis=0) # Combine into PQ image.
    S = tf.expand_dims(S, -1) # Add Channel Dimension.

    # Build FFT
    S_fft = tf.signal.fft2d(tf.cast(S[:,:,0], tf.complex64)) # Compute 2D Fourier transform of PQ image.
    S_fft = tf.abs(S_fft) # Keep magnitude only.
    S_fft = tf.expand_dims(S_fft, -1) # Add Channel Dimension

    return (S, S_fft)

`make_image` converts a batch of input windows into two representations:
- It treats columns 0 and 1 as active power (p) and reactive power (q).
- It builds a pairwise PQ signature matrix, where each cell combines a `p` and `q` value using $\sqrt{p^2+q^2}$, with the original values added along the top and left edges.
- It computes the 2D Fourier Transform of that matrix and keeps only its magnitude.

It returns the PQ signature and its FFT magnitude, each with a channel dimension for model input.

### Generate batches

In [19]:
def generate_sample(x_data, y_data, i_inp, i_out, window_length):

    x_win = x_data[i_inp : i_inp + window_length] # (W,2)
    if x_win.shape[0] != window_length: return None
    y_target = y_data[i_out]
    S, S_fft = make_image(x_win)
    return (S, S_fft), y_target

`generate_sample` creates one training example. 
- Extracts `window_length` rows from `x_data`, starting at `i_inp`, returns `None` if the window is incomplete.
- Selects the corresponding target row from `y_data` at `i_out`.
- Converts the input window into a PQ signature image and its FFT magnitude using `make_image`.
- Returns both images as inputs, paired with the target.

In [20]:
def generate_batch(x_data, y_data, inp_idx, out_idx, window_length):

    S_list = []
    S_fft_list = []
    y_list = []

    for i_inp, i_out in zip(inp_idx, out_idx):
        result = generate_sample(x_data, y_data, i_inp, i_out, window_length)
        if result is None: continue
        (S, S_fft), y = result
        S_list.append(S.numpy()) # (31, 31, 1)
        S_fft_list.append(S_fft[..., np.newaxis])
        y_list.append(y)

    return (np.array(S_list, dtype=np.float32), np.array(S_fft_list, dtype=np.float32)), np.array(y_list, dtype=np.float32)

`generate_batch` builds a batch of model examples from paired input and output indices:
- Iterates through each (`i_inp`, `i_out`) pair and calls `generate_sample`.
- Skips incomplete windows (`None` results).
- Collects each sample's PQ image, FFT image, and target.
- Converts the lists into `float32` NumPy arrays and returns `((PQ_images, FFT_images), targets)`.

So the returned structure is ready to feed into a model with two inputs and corresponding targets.

## Train Model

In [21]:
def train_model(data, idx_dict, window_length, epochs, batch_size, model_filepath):
    
    def build_branch(input_layer):
        x = layers.Conv2D(30, (10, 10), activation='relu')(input_layer)
        x = layers.Conv2D(30, (8, 8), activation='relu')(x)
        x = layers.Conv2D(40, (6, 6), activation='relu')(x)
        x = layers.Conv2D(50, (5, 5), activation='relu')(x)
        x = layers.Conv2D(50, (5, 5), activation='relu')(x)
        x = layers.Flatten()(x)
        return x
    
    # Inputs
    inp_time = layers.Input(shape=(31, 31, 1))
    inp_freq = layers.Input(shape=(31, 31, 1))

    # Branches
    branch_time = build_branch(inp_time)
    branch_freq = build_branch(inp_freq)
    x = layers.Concatenate()([branch_time, branch_freq])

    # Dense + Output
    x = layers.Dense(1024, activation='relu')(x)
    out = layers.Dense(1)(x)

    # Model
    model = models.Model(inputs=[inp_time, inp_freq], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse')

    # Unpack Data
    x_data = data['x_norm']
    y_data = data['y_norm']
    
    train_inp, train_out = idx_dict['train']
    val_inp, val_out = idx_dict['val']

    best_val_loss = np.inf
    patience = 5
    patience_counter = 0
    
    for epoch in tqdm(range(epochs), desc="Epochs"):

        # Training
        train_loss = 0.0
        num_train_batches = 0

        # Shuffle each Epoch
        perm = np.random.permutation(len(train_inp))
        train_inp = train_inp[perm]
        train_out = train_out[perm]

        for i in tqdm(range(0, len(train_inp), batch_size), desc="Training", leave=False):
            batch_inp = train_inp[i:i + batch_size]
            batch_out = train_out[i:i + batch_size]
            (S, S_fft), y = generate_batch(x_data, y_data, batch_inp, batch_out, window_length)

            loss = model.train_on_batch([S, S_fft], y)
            train_loss += loss
            num_train_batches += 1

        train_loss /= num_train_batches

        # Validation
        val_loss = 0.0
        num_val_batches = 0

        for i in tqdm(range(0, len(val_inp), batch_size), desc="Validation", leave=False):
            batch_inp = val_inp[i:i + batch_size]
            batch_out = val_out[i:i + batch_size]
            (S, S_fft), y = generate_batch(x_data, y_data, batch_inp, batch_out, window_length)

            loss = model.test_on_batch([S, S_fft], y)
            val_loss += loss
            num_val_batches += 1

        val_loss /= num_val_batches
        print(f"Epoch {epoch + 1}/{epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

        # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            model.save(model_filepath)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping")
                break

    model.save(model_filepath)

The `train_model` function builds and trains a CNN that uses both the time-domain and ffrequency-domain images:
- **Model**: Two separate convolutional branches process `s` and `s_fft`. Their features are concatenated, passed through a 1024-unit dense layer, and used to predict one value.
- **Setup**: Adam optimizer and mean squared error (MSE) loss are used. The function reads normalized data and training/validation indices from `data` and `idx_dict`.
- **Training**: Each epoch shuffles the training index pairs, generates examples in batches, and updates the model with `train_on_batch`.
- **Validation and stopping**: It evaluates validation batches after each epoch. If validation loss improves, it saves the model. Otherwise, it stops after five consecutive non-improving epochs.

In [22]:
model_filepath = os.path.join(model_dir, "ampds_model.keras")

train_model(
    data=data,
    idx_dict=idx_dict,
    window_length=30,  # Matches the model's 31×31 image input
    epochs=50,
    batch_size=32,
    model_filepath=model_filepath)

Epochs:   0%|          | 0/50 [00:00<?, ?it/s]


ValueError: Exception encountered when calling Functional.call().

[1mInvalid input shape for input Tensor("data:0", shape=(0,), dtype=float32) with name 'keras_tensor' and path '0'. Expected shape (None, 31, 31, 1), but input has incompatible shape (0,)[0m

Arguments received by Functional.call():
  • inputs=['tf.Tensor(shape=(0,), dtype=float32)', 'tf.Tensor(shape=(0,), dtype=float32)']
  • training=True
  • mask=['None', 'None']